# 05 · Multirate processing and resampling

Every real transceiver changes sampling rates.  The DSP shapes symbols at
2 samples per symbol, the DAC and the ADC run at their own fixed rates (often
a non-integer multiple of the symbol rate), and the receiver algorithms
want 2 samples per symbol again, or 1.  This notebook covers:

1. **upsampling** by an integer factor and the spectral images it removes;
2. **decimation** and aliasing;
3. dropping to the **symbol rate** after the matched filter;
4. **rational resampling** to an arbitrary rate;
5. **periodic** records for an arbitrary waveform generator (AWG).

All rate changes keep CommKit's power convention: `E[|x|²] = 1/sps`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)


def host(a):
    return to_device(a, "cpu")


def mean_power(sig):
    return float(np.mean(np.abs(host(sig.samples)) ** 2))

## 1. Upsampling

Upsampling by `L` inserts `L - 1` zeros between samples, which replicates
the spectrum `L` times across the new band (images), and then low-pass
filters the images away.  `upsample` does both in one polyphase pass.  For
a `Signal` it updates `sampling_rate` and rescales the power so that
`E[|x|²] = 1/sps` still holds.

In [ ]:
tx = ck.generate(
    C16,
    num_symbols=2**14,
    symbol_rate=10e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=1,
    device=DEVICE,
)
up = ck.multirate.upsample(tx, factor=4)
print(
    f"sps {tx.sps:.0f} -> {up.sps:.0f}, fs {tx.sampling_rate / 1e9:.0f} -> {up.sampling_rate / 1e9:.0f} GHz"
)
print(
    f"power {mean_power(tx):.4f} -> {mean_power(up):.4f}   (1/sps = {1 / up.sps:.4f})"
)

# What the filter removes: zero-stuffing alone leaves images of the spectrum.
stuffed = tx.xp.zeros(4 * tx.samples.size, dtype=tx.samples.dtype)
stuffed[::4] = tx.samples
fig, ax = plt.subplots(figsize=(8, 4))
ck.plotting.plot_psd(
    stuffed,
    sampling_rate=up.sampling_rate,
    nperseg=1024,
    ax=ax,
    title=None,
    label="zeros inserted (images)",
)
ck.plotting.plot_psd(
    up,
    nperseg=1024,
    ax=ax,
    title="Upsampling 2 -> 8 samples per symbol",
    label="upsample()",
)
ax.legend()
plt.show()

## 2. Decimation and aliasing

Keeping every `M`-th sample folds everything above the new Nyquist
frequency `fs / (2M)` back into the band.  `decimate` filters first (an
anti-aliasing low-pass), then drops samples.  Here an interferer at 25 GHz,
outside the band of a 2-sample-per-symbol receiver, folds onto the signal
when the samples are simply sliced:

In [ ]:
x8 = ck.spectral.add_pilot_tone(up, frequency=25e9, power_ratio_db=0)
# Slicing alone; the factor 2 = sqrt(4) restores E|x|^2 = 1/sps, as decimate() does.
sliced = x8.replace(
    samples=2 * x8.samples[..., ::4], sampling_rate=x8.sampling_rate / 4
)
filtered = ck.multirate.decimate(x8, factor=4)

fig, ax = plt.subplots(figsize=(8, 4))
ck.plotting.plot_psd(
    sliced,
    nperseg=512,
    ax=ax,
    title=None,
    label="sliced: tone aliased to 25 - 20 = 5 GHz",
)
ck.plotting.plot_psd(
    filtered,
    nperseg=512,
    ax=ax,
    title="Decimating 8 -> 2 samples per symbol",
    label="decimate(): tone removed",
)
ax.legend()
plt.show()

## 3. To the symbol rate

After the matched filter the waveform is ISI-free at one sampling phase,
and the receiver keeps one sample per symbol.  This is not a filtering
decimation: the matched filter already limited the noise, and slicing at
the right phase is the whole point.  `decimate_to_symbol_rate` slices (with
`offset=` selecting the phase) and keeps the reference aligned; at one
sample per symbol the power is `E[|x|²] = 1`.

In [ ]:
mf = ck.filtering.matched_filter(tx)
y = ck.multirate.decimate_to_symbol_rate(mf)
print(
    f"sps {y.sps:.0f}, power {mean_power(y):.3f}, {y.samples.shape[-1]} samples, "
    f"{y.reference.symbols.shape[-1]} reference symbols"
)
print(
    f"signal-to-ISI {ck.metrics.snr(y):.1f} dB at offset 0, "
    f"{ck.metrics.snr(ck.multirate.decimate_to_symbol_rate(mf, offset=1)):.1f} dB at offset 1"
)

## 4. Rational resampling

A DAC or ADC rarely runs at an integer multiple of the symbol rate: a 10 GBd
signal on a 25 GS/s converter has 2.5 samples per symbol.  `resample`
changes the rate by `up / down` with one polyphase filter, or to a target
`sps_out` (the ratio becomes a fraction).  A round trip back to 2 samples
per symbol shows the interpolation error, limited by the filter's stopband.

In [ ]:
adc = ck.multirate.resample(tx, sps_out=2.5)
back = ck.multirate.resample(adc, sps_out=2)
print(
    f"{tx.sampling_rate / 1e9:.0f} GS/s -> {adc.sampling_rate / 1e9:.0f} GS/s -> {back.sampling_rate / 1e9:.0f} GS/s"
)
print(
    f"power after each step: {mean_power(tx):.4f}, {mean_power(adc):.4f}, {mean_power(back):.4f}"
)

a, b = host(tx.samples), host(back.samples)
inner = slice(500, -500)  # the filter transients at the edges are excluded
err = np.mean(np.abs(a[inner] - b[inner]) ** 2) / np.mean(np.abs(a[inner]) ** 2)
print(f"round-trip error {10 * np.log10(err):.1f} dB relative to the signal")

z = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(back))
print(f"SNR of the round trip at the symbol instants: {ck.metrics.snr(z):.1f} dB")

## 5. Periodic records for an AWG

A streaming AWG plays its memory in a loop.  If the record was shaped by a
linear convolution, its two ends carry filter transients, and the loop seam
is a discontinuity that corrupts the symbols around it.  `periodic=True`
shapes by **circular** convolution, so the record is exactly one period of
a periodic waveform and the seam is like any other point.

Play each record four times, matched-filter, and compare the worst error
at a seam with the worst error inside:

In [ ]:
n = 2**12
for periodic in (False, True):
    awg = ck.generate(
        C16,
        num_symbols=n,
        symbol_rate=32e9,
        sps=2,
        pulse=RRC(rolloff=0.1),
        periodic=periodic,
        rng=6,
        device=DEVICE,
    )
    xp = awg.xp
    played = ck.filtering.matched_filter(
        xp.tile(awg.samples, 4), pulse=awg.pulse.taps(2)
    )
    y4 = host(played[::2])
    ref = np.tile(host(awg.reference.symbols), 4)
    err = np.abs(y4 * np.vdot(ref, ref) / np.vdot(ref, y4) - ref)
    print(
        f"periodic={periodic!s:5}: worst error at a seam {err[n - 8 : n + 8].max():.3f}, "
        f"inside {err[n + 100 : 2 * n - 100].max():.3f}"
    )

The AWG runs at its own fixed rate, here 92 GS/s.  Resampling must keep
the record periodic too: `resample(..., periodic=True)` interpolates in the
frequency domain and returns exactly one period at the new rate (the new
length must be an integer).  Every transmitter step after `generate` takes
`periodic=True` explicitly.

In [ ]:
dac = ck.multirate.resample(awg, sps_out=92e9 / awg.symbol_rate, periodic=True)
print(
    f"{dac.samples.shape[-1]} samples at {dac.sampling_rate / 1e9:.0f} GS/s "
    f"({dac.sps:.4f} samples per symbol)"
)

**Summary**

* `upsample` / `decimate` change the rate by an integer with the filtering
  that prevents images and aliasing; `resample` handles any rational ratio.
* `decimate_to_symbol_rate` is the slicer after the matched filter, and keeps
  the reference aligned.
* Rate changes on a `Signal` update `sampling_rate` and keep
  `E[|x|²] = 1/sps`.
* For AWG records, shape and resample with `periodic=True`.

**Next:** [06 · AWGN and performance metrics](06-awgn-and-performance-metrics.ipynb).